<div style="width: 100%; text-align: center;">
    <div style="background-color:#007F00; padding: 0.5rem;">
        <h1 style="font-weight: bold; font-size: 2.5em; color: black;"> AGRHYMET CENTRE CLIMATIQUE REGIONAL POUR L'AFRIQUE DE L'OUEST ET LE SAHEL</h1>
   </div>

   <div style="text-align: center;">
  <img src="https://www.sareco.org/wp-content/uploads/2017/07/plrDvYX1.jpg" width="200">
</div>

<a id="1"></a>
### <p style="padding:10px;background-color:#000000 ;margin:0;color:#007F00;font-family:#newtimeroman;font-size:100%;text-align:center;border-radius: 15px 50px;overflow:hidden;font-weight:500"> Download GloFas Data </p>

In [ ]:
import sys, os
from pathlib import Path
from importlib.metadata import version
from glofas_download import download_glofas_discharge
from glofas_extract import extract_glofas_at_points, read_points_csv
from glofas_visualize import build_timeseries_figure, build_points_map

In [ ]:
ewds_config = Path.home() / ".cdsapirc-ewds"

if not ewds_config.exists():
    raise FileNotFoundError(
        f"Configuration EWDS introuvable : {ewds_config}"
    )

os.environ["CDSAPI_RC"] = str(ewds_config)

print("Configuration sélectionnée :", ewds_config)

In [ ]:
results = download_glofas_discharge(
    year=range(1980, 1984),
    months=range(1, 13),
    area=(7.0, 8.0, 2.0, 16.0),
    output_dir="glofas_data",
    pause_between_years=5,
)

## Extraction ponctuelle aux points d'intérêt

Une fois les fichiers mensuels téléchargés ci-dessus, on peut en extraire
des séries temporelles à des coordonnées précises grâce à
`glofas_extract.py`. L'utilisateur fournit un CSV avec les colonnes
`ID`, `LONG`, `LAT` (d'autres noms de colonnes usuels sont détectés
automatiquement).

Comme ces coordonnées ne tombent pas toujours exactement sur une maille
du réseau hydrographique simulé par GloFAS, on peut élargir la recherche
avec `radius_km` : la maille retenue est alors celle qui présente le
débit le plus élevé (donc la plus probable d'appartenir à un chenal)
parmi toutes les mailles comprises dans ce rayon. Avec `radius_km=0`, on
se limite à la maille géographiquement la plus proche.


Fichier de points fourni par l'utilisateur, par exemple :

| ID | LONG | LAT |
|----|------|-----|
| S01 | 11.52 | 3.87 |
| S02 | 13.68 | 4.05 |
| S03 | 9.90  | 5.96 |


In [ ]:
points_csv = "points_exemple.csv"  # colonnes ID, LONG, LAT

# Vérification rapide du fichier avant extraction (détection des colonnes,
# validation des coordonnées).
read_points_csv(points_csv)


In [ ]:
series = extract_glofas_at_points(
    points=points_csv,
    input_dir="glofas_data",       # dossier des fichiers téléchargés plus haut
    start="1980-01",               # période à extraire (bornée par les données disponibles)
    end="1983-12",
    radius_km=10.0,                # rayon de recherche autour de chaque point (km)
    method="max",                  # 'max' = recalage sur le débit le plus fort dans le rayon
    agg_stat="mean",               # statistique utilisée pour choisir la maille ('mean' ou 'max')
    output="resultats/extraction", # écrit *_series.csv et *_points.csv
    make_report=True,              # + carte interactive et rapport HTML (voir plus bas)
)

series.head()


### Vérifier le recalage spatial

Le fichier `*_points.csv` (aussi accessible via `series` groupé par `id`)
indique, pour chaque point : la maille retenue (`lon_pixel`, `lat_pixel`),
la distance au point d'origine (`distance_km`) et le statut du recalage
(`statut` : `ok` = maille la plus proche, `recale` = maille recalée dans le
rayon, `repli` = aucune maille valide trouvée dans le rayon, repli sur la
plus proche). C'est ce même statut qui colore les points sur la carte
ci-dessous.


In [ ]:
series.drop_duplicates("id")[["id", "lon_input", "lat_input", "lon_pixel", "lat_pixel", "distance_km"]]


## Visualiser les résultats

`make_report=True` ci-dessus a déjà généré trois fichiers HTML dans
`resultats/` (`extraction_carte.html`, `extraction_series.html`,
`extraction_rapport.html`). On peut aussi afficher la carte et le
graphique directement dans le notebook avec `glofas_visualize`, en
choisissant les stations et la période à afficher — soit par code
(`ids=...`, `start=...`, `end=...`), soit avec un petit tableau de bord
interactif qui permet aussi d'enregistrer le graphique affiché.

*La carte a besoin d'une connexion Internet pour charger les fonds de
carte (OpenStreetMap) — normalement disponible puisque le téléchargement
GloFAS lui-même nécessite l'accès à l'EWDS. Le graphique des séries
temporelles fonctionne, lui, entièrement hors connexion.*


In [ ]:
#build_points_map("resultats/extraction_points.csv")  # affichage interactif dans le notebook

In [ ]:
# Toutes les stations, toute la période :
fig = build_timeseries_figure("resultats/extraction_series.csv")
fig.show()

# Une sélection de stations et une période précises, avec sauvegarde :
fig_filtre = build_timeseries_figure(
    "resultats/extraction_series.csv",
    ids=["S01", "S02"],           # None ou omis = toutes les stations
    start="1981-01", end="1981-12",
    output_html="resultats/extraction_series_1981_S01_S02.html",  # None = pas d'enregistrement
)
fig_filtre.show()


### Tableau de bord interactif

Pour choisir les stations et la période sans réécrire de code à chaque
fois — utile en démonstration ou pour un·e participant·e moins à l'aise
avec Python — `interactive_timeseries_explorer` affiche une liste de
sélection des stations, deux sélecteurs de date, et un bouton
« Enregistrer le graphique » qui écrit le graphique actuellement affiché
vers le fichier indiqué (modifiable). Nécessite `ipywidgets` (inclus dans
l'environnement Conda de l'atelier).


In [ ]:
from glofas_visualize import interactive_timeseries_explorer

interactive_timeseries_explorer("resultats/extraction_series.csv")
